<a href="https://colab.research.google.com/github/marcnasrisme/NLP-Project/blob/main/notebooks/02_train_adapters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 - Train Emotion-Specific QLoRA Adapters

Trains 4 QLoRA adapters on Mistral-7B-Instruct, one per emotion cluster.

**Runtime:** set to GPU (A100) + High RAM in Colab via Runtime > Change runtime type.

## 1. Setup

In [1]:
# clone repo and install deps
import os

if not os.path.exists("/content/NLP-Project/src"):
    !git clone https://github.com/marcnasrisme/NLP-Project.git /content/NLP-Project
else:
    !cd /content/NLP-Project && git pull

%cd /content/NLP-Project
!pip install -q -r requirements.txt

Cloning into '/content/NLP-Project'...
remote: Enumerating objects: 79, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 79 (delta 33), reused 74 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (79/79), 166.06 KiB | 41.52 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/content/NLP-Project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 23.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [2]:
# mount google drive for persistent checkpoints
from google.colab import drive
drive.mount('/content/drive')

ADAPTER_OUTPUT = "/content/drive/MyDrive/NLP_Project_adapters"
os.makedirs(ADAPTER_OUTPUT, exist_ok=True)
print(f"adapters will be saved to: {ADAPTER_OUTPUT}")

Mounted at /content/drive
adapters will be saved to: /content/drive/MyDrive/NLP_Project_adapters


In [3]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


## 2. Load and prepare data

In [4]:
from src.data.load import load_splits
from src.data.cluster import build_emotion_clusters, save_clusters
from src.data.format import format_for_sft

splits = load_splits()
print(f"loaded: {len(splits['train'])} train, {len(splits['validation'])} val, {len(splits['test'])} test conversations")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/76673 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/12030 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10943 [00:00<?, ? examples/s]

loaded: 17844 train, 2763 val, 2542 test conversations


In [5]:
# cluster emotions and save config
cluster_result = build_emotion_clusters()
save_clusters(cluster_result, "configs/emotion_clusters.yaml")

emotion_to_cluster = cluster_result["emotion_to_cluster"]
cluster_names = cluster_result["cluster_names"]

for cid, name in cluster_names.items():
    emotions = cluster_result["clusters"][name]["emotions"]
    print(f"Cluster {cid} ({name}): {len(emotions)} emotions")

Cluster 2 (positive_low_arousal): 5 emotions
Cluster 1 (positive_high_arousal): 12 emotions
Cluster 3 (negative_low_arousal): 7 emotions
Cluster 0 (negative_high_arousal): 8 emotions


In [6]:
# format all training examples
train_examples = format_for_sft(splits["train"], emotion_to_cluster)
print(f"total training examples: {len(train_examples)}")

# split by cluster
cluster_data = {}
for cid, name in cluster_names.items():
    cluster_data[cid] = [ex for ex in train_examples if ex["cluster_id"] == cid]
    print(f"  cluster {cid} ({name}): {len(cluster_data[cid])} examples")

total training examples: 35459
  cluster 2 (positive_low_arousal): 5153 examples
  cluster 1 (positive_high_arousal): 13269 examples
  cluster 3 (negative_low_arousal): 7601 examples
  cluster 0 (negative_high_arousal): 9436 examples


## 3. Load base model

This loads Mistral-7B-Instruct in 4-bit. Takes ~2 minutes on first run (downloads ~14.5GB, quantized to ~5GB in memory).

In [7]:
from src.models.adapter import load_config, load_base_model

config = load_config("configs/adapter_training.yaml")
base_model, tokenizer = load_base_model(config)
print(f"model loaded: {config['model']['name']}")
print(f"tokenizer vocab size: {len(tokenizer)}")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model loaded: mistralai/Mistral-7B-Instruct-v0.3
tokenizer vocab size: 32768


## 4. Train adapters

Trains one adapter per cluster, saved to Google Drive.

If Colab dies mid-training, just re-run all cells from the top. Training resumes from the last checkpoint on Drive.

**To train specific clusters**, change `clusters_to_train` below (e.g. `[1, 2, 3]` to skip cluster 0).

In [20]:
# change this to train specific clusters, e.g. [1, 2, 3] to skip already-trained ones
# set to None to train all 4
clusters_to_train = [2,3]

In [21]:

  import torch
  import src.models.adapter as _adapter
  from trl import SFTConfig

  def _fixed_make_training_args(config, output_dir):
      train = config["training"]
      use_bf16 = train.get("bf16", False) and torch.cuda.is_available()
      return SFTConfig(
          output_dir=output_dir,
          num_train_epochs=train["epochs"],
          per_device_train_batch_size=train["batch_size"],
          gradient_accumulation_steps=train["gradient_accumulation_steps"],
          learning_rate=train["learning_rate"],
          warmup_ratio=train["warmup_ratio"],
          save_steps=train["save_steps"],
          logging_steps=train["logging_steps"],
          bf16=use_bf16,
          save_total_limit=2,
          report_to="none",
      )

  _adapter.make_training_args = _fixed_make_training_args
  print("patched")


patched


In [22]:

from src.models.adapter import attach_adapter, train_adapter

target_clusters = clusters_to_train or list(cluster_names.keys())

for cid in target_clusters:
    name = cluster_names[cid]
    output_dir = f"{ADAPTER_OUTPUT}/cluster_{cid}_{name}"
    examples = cluster_data[cid]

    print(f"\n{'='*60}")
    print(f"Training adapter for cluster {cid}: {name}")
    print(f"  examples: {len(examples)}")
    print(f"  output: {output_dir}")
    print(f"{'='*60}\n")

    model = attach_adapter(base_model, config)

    train_adapter(
        model=model,
        tokenizer=tokenizer,
        train_examples=examples,
        output_dir=output_dir,
        config=config,
    )

    base_model = model.unload()
    print(f"\nAdapter {cid} ({name}) saved to {output_dir}")


Training adapter for cluster 2: positive_low_arousal
  examples: 5153
  output: /content/drive/MyDrive/NLP_Project_adapters/cluster_2_positive_low_arousal



/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Converting train dataset to ChatML:   0%|          | 0/5153 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/5153 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5153 [00:00<?, ? examples/s]

...

Step,Training Loss
10,3.605110
20,2.512303
30,2.017590
40,1.838131
50,1.705518
60,1.693791
70,1.620111
80,1.582221
90,1.611604
100,1.600108


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

Step,Training Loss
10,3.605110
20,2.512303
30,2.017590
40,1.838131
50,1.705518
60,1.693791
70,1.620111
80,1.582221
90,1.611604
100,1.600108


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt


Adapter 3 (negative_low_arousal) saved to /content/drive/MyDrive/NLP_Project_adapters/cluster_3_negative_low_arousal


## 5. Verify saved adapters

In [26]:
# checking they all really ran
import os
import json

for cid, name in cluster_names.items():
    adapter_dir = f"{ADAPTER_OUTPUT}/cluster_{cid}_{name}"
    print(f"\n{'='*50}")
    print(f"Cluster {cid}: {name}")
    print(f"{'='*50}")

    if not os.path.exists(adapter_dir):
        print("  NOT FOUND")
        continue

    # check main dir for trainer_state
    state_file = os.path.join(adapter_dir, "trainer_state.json")

    # if not in main dir, check the last checkpoint
    if not os.path.exists(state_file):
        checkpoints = sorted(
            [d for d in os.listdir(adapter_dir) if d.startswith("checkpoint-")],
            key=lambda x: int(x.split("-")[1])
        )
        if checkpoints:
            state_file = os.path.join(adapter_dir, checkpoints[-1], "trainer_state.json")

    if os.path.exists(state_file):
        with open(state_file) as f:
            state = json.load(f)
        epoch = state.get("epoch", "?")
        step = state.get("global_step", "?")
        max_steps = state.get("max_steps", "?")
        # get final training loss from log history
        log_history = state.get("log_history", [])
        losses = [l["loss"] for l in log_history if "loss" in l]
        last_loss = f"{losses[-1]:.4f}" if losses else "?"
        print(f"  epoch:     {epoch} / 3.0")
        print(f"  steps:     {step} / {max_steps}")
        print(f"  last loss: {last_loss}")
        done = (epoch == 3.0 or step == max_steps) if isinstance(epoch, (int, float)) else "unknown"
        print(f"  complete:  {done}")
    else:
        print("  no trainer_state.json found - cannot verify")

    # list files in main dir
    files = os.listdir(adapter_dir)
    has_weights = "adapter_model.safetensors" in files or "adapter_model.bin" in files
    print(f"  weights:   {'yes' if has_weights else 'NO'}")



Cluster 2: positive_low_arousal
  epoch:     3.0 / 3.0
  steps:     969 / 969
  last loss: 0.8284
  complete:  True
  weights:   yes

Cluster 1: positive_high_arousal
  epoch:     3.0 / 3.0
  steps:     2490 / 2490
  last loss: 0.9144
  complete:  True
  weights:   yes

Cluster 3: negative_low_arousal
  epoch:     3.0 / 3.0
  steps:     1428 / 1428
  last loss: 0.7935
  complete:  True
  weights:   yes

Cluster 0: negative_high_arousal
  epoch:     3.0 / 3.0
  steps:     1770 / 1770
  last loss: 0.8347
  complete:  True
  weights:   yes


In [23]:
for cid, name in cluster_names.items():
    adapter_dir = f"{ADAPTER_OUTPUT}/cluster_{cid}_{name}"
    if os.path.exists(adapter_dir):
        files = os.listdir(adapter_dir)
        has_weights = "adapter_model.safetensors" in files or "adapter_model.bin" in files
        has_config = "adapter_config.json" in files
        status = "ready" if (has_weights and has_config) else "incomplete"
        print(f"Cluster {cid} ({name}): {status}")
    else:
        print(f"Cluster {cid} ({name}): not trained yet")

Cluster 2 (positive_low_arousal): ready
Cluster 1 (positive_high_arousal): ready
Cluster 3 (negative_low_arousal): ready
Cluster 0 (negative_high_arousal): ready


## 6. Quick test: generate with one adapter

In [28]:
from src.models.adapter import load_trained_adapter

test_cid = target_clusters[0]
test_name = cluster_names[test_cid]
adapter_path = f"{ADAPTER_OUTPUT}/cluster_{test_cid}_{test_name}"

model_with_adapter = load_trained_adapter(base_model, adapter_path)
model_with_adapter.eval()

prompt = "[INST] Situation: I just lost my job.\nEmotion: devastated\n\nSpeaker: I got laid off today. I don't know what to do. [/INST]"
inputs = tokenizer(prompt, return_tensors="pt").to(model_with_adapter.device)

with torch.no_grad():
    output = model_with_adapter.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True,
    )

response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"Adapter: {test_name}")
print(f"Response: {response}")

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adapter: positive_low_arousal
Response: I am so sorry. This is a common thing and it will happen to everyone. You will get another job and you will be fine.


In [30]:
# testing base model against each adapter and for different prompts

from src.models.adapter import load_trained_adapter

test_prompts = {
    0: "[INST] Situation: My house was broken into.\nEmotion: terrified\n\nSpeaker: Someone broke into my house last night. [/INST]",
    1: "[INST] Situation: I got into my dream school.\nEmotion: joyful\n\nSpeaker: I just got my acceptance letter! [/INST]",
    2: "[INST] Situation: I found an old photo album.\nEmotion: nostalgic\n\nSpeaker: I found photos from when I was a kid. [/INST]",
    3: "[INST] Situation: I failed my exam.\nEmotion: disappointed\n\nSpeaker: I studied so hard but I still failed. [/INST]",
}

def generate_response(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

# test base model (no adapter)
print("=" * 60)
print("BASE MODEL (no adapter)")
print("=" * 60)
for cid, prompt in test_prompts.items():
    print(f"\nPrompt ({cluster_names[cid]}):")
    print(f"  {prompt.split('Speaker: ')[1].replace(' [/INST]', '')}")
    print(f"  Response: {generate_response(base_model, prompt)}")

# test each adapter
for cid, name in cluster_names.items():
    print(f"\n{'=' * 60}")
    print(f"ADAPTER {cid}: {name}")
    print(f"{'=' * 60}")

    adapter_path = f"{ADAPTER_OUTPUT}/cluster_{cid}_{name}"
    model_with_adapter = load_trained_adapter(base_model, adapter_path)
    model_with_adapter.eval()

    for pid, prompt in test_prompts.items():
        print(f"\nPrompt ({cluster_names[pid]}):")
        print(f"  {prompt.split('Speaker: ')[1].replace(' [/INST]', '')}")
        print(f"  Response: {generate_response(model_with_adapter, prompt)}")

    base_model = model_with_adapter.unload()


BASE MODEL (no adapter)

Prompt (negative_high_arousal):
  Someone broke into my house last night.
  Response: Were they able to find any evidence?

Prompt (positive_high_arousal):
  I just got my acceptance letter!
  Response: Congrats! For what? College?

Prompt (positive_low_arousal):
  I found photos from when I was a kid.
  Response: how are you feeling about it

Prompt (negative_low_arousal):
  I studied so hard but I still failed.
  Response: Failed what?

ADAPTER 2: positive_low_arousal

Prompt (negative_high_arousal):
  Someone broke into my house last night.
  Response: Someone broke into my house last night. I'm so shocked!

Prompt (positive_high_arousal):
  I just got my acceptance letter!
  Response: Congrats! For what? College?

Prompt (positive_low_arousal):
  I found photos from when I was a kid.
  Response: That's cool. Did you go through them and remember all the good times?

Prompt (negative_low_arousal):
  I studied so hard but I still failed.
  Response: That's awf